# పాఠం 10 - ఉత్పత్తిలో AI ఏజెంట్లు

ఈ పాఠంలో మీరు Microsoft Agent Framework (Python) ఉపయోగించి AI ఏజెంట్లకు **ఉత్పత్తి నమూనాలు** నేర్చుకుంటారు. మనం ఈ విషయాలు కవర్ చేస్తాము:

- **పర్యవేక్షణ** — ఏజెంట్ ఇంటరాక్షన్‌లకు టైమింగ్ మరియు లాగింగ్ జోడించడం
- **మూల్యాంకనం** — స్పందన నాణ్యతను స్కోర్ చేయడానికి ఒక మూల్యాంకక ఏజెంట్ ఉపయోగించడం
- **ఖర్చు నిర్వహణ** — టోకెన్ ఆప్టిమైజేషన్ మరియు మోడల్ ఎంపిక కోసం వ్యూహాలు

సందర్భం ఒక **ప్రయాణ ఏజెంట్** గురించి, ఇది వినియోగదారులకు ప్రయాణాలు ప్లాన్ చేయడంలో సహాయపడుతుంది, దాని పై పర్యవేక్షణ మరియు మూల్యాంకనం ఆకృతులు అమర్చబడ్డాయి.


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ఉత్పత్తి పరిగణనలు

AI ఏజెంట్లను ప్రోటోటైప్ల నుండి ఉత్పత్తికి తీసుకెళ్లడం మూడు ప్రధానాంశాలకు జాగ్రత్తగా శ్రద్ధ అవసరం:

1. **నిరీక్షణ** — ఏజెంట్ ఏమి చేస్తోంది, ఎంతసేపు తీసుకుంటోంది, ఏ సాధనాలను పిలుస్తోంది అన్న దానిపై మీకు దృష్టి అవసరం. ట్రేసింగ్ మరియు లాగింగ్ లేకపోతే ఉత్పత్తి సమస్యలను డీబగ్ చేయడం సుమారు అసాధ్యం.

2. **అంచనా** — ఆటోమేటెడ్ నాణ్యత చూపులతో ఏజెంట్ సమాధానాలు సమయానికి సరైనవిగా, పూర్తిగా, సాయపడగలిగేవిగా ఉండడం నిర్ధారించాలి. అంచనా ముఖ్యుడు ఏజెంట్ నిర్వచిత ప్రమాణాలపై సమాధానాలను స్కోర్ చేయగలడు.

3. **ఖర్చు నిర్వహణ** — టోకన్ల వాడకం నేరుగా ఖర్చుపై ప్రభావం చూపుతుంది. ప్రాంప్ట్ ఆప్టిమైజేషన్, మోడల్ ఎంపిక, మరియు కాచింగ్ వంటి వ్యూహాలు నాణ్యతను కోల్పోకుండా ఖర్చులను నియంత్రణలో ఉంచడానికి సహాయపడతాయి.


## ఒక దర్శనీయ ఏజెంట్‌ను నిర్మించడం

మేము ప్రయాణ సాధనాలను నిర్వచించి, ఆలస్యాన్ని పర్యవేక్షించడానికి ఏజెంట్ కాల్‌ను టైమింగ్‌తో చుట్టింవుతాము. ఉత్పత్తిలో మీరు OpenTelemetry లేదా దీని వంటి ట్రేసింగ్ బ్యాక్‌ఎండ్‌తో సమీకరించేవారు.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## మూల్యాంకన నమూనాలు

సాధారణ ఉత్పత్తి నమూనా రెండవ ఏజెంట్‌ను **మూల్యాంకకుడిగా** ఉపయోగించడం. మూల్యాంకకుడు ప్రాథమిక ఏజెంట్ యొక్క స్పందనను పూర్తి చేయడం, నిర్ధారణ మరియు సహాయకత్వం వంటి ముందుగా నిర్ణయించిన ప్రమాణాలపై స్కోర్ చేస్తుంది.

ఇది అనుమతిస్తుంది:
- వాడుకరులకు స్పందనలు చేరుకునే ముందు ఆటోమేటెడ్ నాణ్యత గేట్లు
- ప్రాంప్టులు లేదా మోడల్స్ మారినప్పుడు రీగ్రెషన్ గుర్తింపు
- సమయానుకూలంగా ఏజెంట్ పనితీరు నిరంతరంగా పర్యవేక్షణ


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## వ్యయ నిర్వహణ వ్యూహాలు

ఉత్పత్తి AI ఏజెంట్‌లకు ఖర్చులను నియంత్రించడం చాలా ముఖ్యం. ముఖ్య వ్యూహాలు ఇవి:

| వ్యూహం | వివరణ |
|---|---|
| **ప్రాంప్ట్ ఆప్టిమైజేషన్** | సిస్టమ్ సూచనలు సంక్షిప్తంగా ఉంచండి. ప్రవేశ టోకెన్లను తగ్గించేందుకు అవసరంలేని సందర్భం తీసివేయండి. |
    "| **మోడల్ ఎంపిక** | వర్గీకరణ లేదా తీయడం వంటి సరళ పనులకు చిన్న, చౌకైన మోడల్స్ (ఉదా: GPT-5-mini) ఉపయోగించండి, కఠినమైన తర్కానికి పెద్ద మోడల్స్ ఉపయోగించండి. |\n",
| **కeshireం** | పునరావృత API కాల్స్ నివారించడానికి సాధన ఫలితాలు మరియు తరచూ వేసే ప్రశ్నలను కeshireం చేయండి. |
| **టోకెన్ బడ్జెట్లు** | అనూహ్యంగా పొడవైన స్పందనలను నివారించేందుకు `max_tokens` పరిమితులు నిర్దేశించండి. |
| **బ్యాట్చింగ్** | ఎక్కడైన అనేక వినియోగదారు ప్రశ్నలను ఒక API కాల్‌లో సమూహీకరించండి. |

ఆచరణలో, శ్రేణీకృత విధానం బాగా పనిచేస్తుంది: సరళమైన అభ్యర్థనలను వేగవంతమైన, చౌకైన మోడల్‌కు పంపించి, కఠినమైన ప్రశ్నలకు మాత్రమే నిపుణులైన మోడల్‌కు ఎక్కించండి.


## సారాంశం

ఈ పాఠంలో మీరు ఎలా చేయాలో నేర్చుకున్నారు:

1. ఏజెంట్ పరస్పర చర్యలకు సమయ నిర్ధారణ మరియు లాగింగ్ తో **నిరీక్షణ సామర్థ్యాన్ని జోడించడం**, ట్రేసింగ్ మరియు మానిటరింగ్ కోసం పునాది వేసుకోవడం.
2. **ఏజెంట్ స్పందనలను** స్వయంచాలకంగా మూల్యాంకితం చేయడం, పూర్తి స్థాయి, సరైనదనం మరియు సహాయపడే లక్షణాలను స్కోరు చేసే మూల్యాంకక ఏజెంట్ ఉపయోగించి.
3. ప్రాంప్ట్ ఆప్టిమైజేషన్, మోడల్ ఎంపిక, కాపింగ్ మరియు టోకెన్ బడ్జెట్ల ద్వారా **ఖర్చులను నిర్వహించడం**.

ఈ ఉత్పత్తి ప్యాటర్న్లు మీ AI ఏజెంట్లు విశ్వసనీయమైన, కొలిచుకోగలిగే, మరియు వ్యయ సామర్థ్యవంతమైనవి కావడానికి సహాయపడతాయి.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
